# PCG Convergence Study  
Using NGSolve's built-in pcg solver and our finite element space hierarchy  


In [ ]:
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt


In [2]:
from src import multigrid_cycles
from src import NGSolve_utils

ModuleNotFoundError: No module named 'src'

In [ ]:
def cmap_to_hex_list(cmap_name, samples=256):
    cmap = plt.get_cmap(cmap_name)
    xs = np.linspace(0, 1, samples)
    return [mcolors.to_hex(cmap(x)) for x in xs]

color_hexes = cmap_to_hex_list(twilight_shifted)

In [ ]:
# Setting up the problem we will solve on every level

# Boundary conditionis
DIRICHLET = "left|right"

# LHS bilinear form
def poisson_bilinear(a, u, v):
    a += InnerProduct(grad(u), grad(v)) * dx

# RHS linear form
rhs_cf = 0
def poisson_linear(f, u, v):
    f += rhs_cf * v * dx

# Initial iterate
x0 = CoefficientFunction(sin(pi*x)*sin(pi*y)+(1/10)*sin(10*pi*x)*sin(10*pi*y))


In [ ]:

# Call our setup function to put these together
poisson_setup = build_form_setup(bilinear=poisson_bilinear, linear=poisson_linear)

# Define the coarsest mesh for our hierarchy of many sized meshes
N = 16
coarsest_mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
parfait = build_hierarchy(
    coarsest_mesh,
    poisson_setup,
    n_refines=4,
    order=1,
    dirichlet=DIRICHLET,
    dirichlet_value={"left": 0.0, "right":0.0},
    verbose=True,
)


In [ ]:
finest = parfait.finest
finest.set_initial_guess(x0)

s = Draw(
    finest.gfu,
    finest.mesh,
    "initial guess (finest)",
    deformation=True,
    settings={"camera":{"transformations": [{"type": "rotateX", "angle": -45}]}},
)